In [ ]:
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# CuPy setup
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
    except Exception as e:
        print(f"✗ Could not load modules: {e}")

from src.utils.array_backend import np, random, is_cupy
from src.belief_quantized.belief_mdp_n_M import BeliefMDP_n_M_Localization
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time
import warnings

warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

"""
Generate p_n^{(M)} transition probability matrix for BeliefMDP_n_M_Localization.

This notebook:
1. Creates a BeliefMDP_n_M_Localization instance with coarse quantization (n=2)
2. Automatically computes and caches p_n^{(M)} for all belief-action pairs
3. Verifies the computed matrix properties

For Localization:
- Belief space: π(x) - shape (m_n,)
- Quantized belief space: Π_n^(M) with N_n = m_n
- Map is known (not part of belief state)
"""


In [ ]:
# Configuration
n = 2  # Coarse quantization for initial testing
M = 2  # Belief space quantization parameter
beta = 0.95  # Discount factor

print(f"Configuration:")
print(f"  n (state quantization): {n}")
print(f"  M (belief quantization): {M}")
print(f"  β (discount factor): {beta}")
print(f"\nThis will create a BeliefMDP_n_M_Localization instance which will:")
print(f"  1. Load or generate belief codebook (Π_n^M) with N_n = m_n")
print(f"  2. Load cached T_mat")
print(f"  3. Compute p_n^{(M)} for all belief-action pairs")
print(f"  4. Cache the result for future use")

# Load environment configuration
obstacles, area = load_obstacles_config(environment='toy2')

# Create models
motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
sensor = LIDAR(fov=360, r_max=10.0, B=8)
grid_map = LidarGridMapVec(
    x_min=area[0], x_max=area[1],
    y_min=area[2], y_max=area[3],
    quantization_level=n,
)

print(f"Environment setup:")
print(f"  Map bounds: x=[{area[0]}, {area[1]}], y=[{area[2]}, {area[3]}]")
print(f"  Motion model: DoubleIntegratorModel (dt={motion_model.dt}, max_a={motion_model.max_a})")
print(f"  Sensor: LIDAR (fov={sensor.fov}°, r_max={sensor.r_max}, B={sensor.B})")


In [ ]:
# Create BeliefMDP_n_M_Localization instance
print("="*70)
print("Creating BeliefMDP_n_M_Localization instance...")
print("="*70)
print()

start_time = time.time()

bmdp_M = BeliefMDP_n_M_Localization(
    M=M,
    β=beta,
    n=n,
    motion_model=motion_model,
    measurement_model=sensor,
    obstacles=obstacles,
    _map=grid_map,
    sigma_v=1.0
)

# Set known map for localization
bmdp_M.set_known_map(obstacles)

elapsed_time = time.time() - start_time

print()
print("="*70)
print("Initialization complete!")
print("="*70)
print(f"Total time: {elapsed_time:.2f}s")
print()
print(f"Belief space statistics:")
print(f"  Cardinality |Π_n^M|: {bmdp_M.BQ.cardinality:,}")
print(f"  Belief dimension N_n: {bmdp_M.SQ.m_n} (for localization, N_n = m_n)")
print(f"  State space size m_n: {bmdp_M.SQ.m_n}")
print(f"  Action space size n_u: {bmdp_M.AQ.n_u}")

# Check if sparse format
is_sparse = isinstance(bmdp_M.p_n_M, list)
if is_sparse:
    print(f"  p_n^{(M)} format: Sparse (CSR, float16)")
    print(f"    - {len(bmdp_M.p_n_M)} actions")
    print(f"    - Each matrix: {bmdp_M.p_n_M[0].shape[0]:,} × {bmdp_M.p_n_M[0].shape[1]:,}")
    total_nnz = sum(mat.nnz for mat in bmdp_M.p_n_M)
    total_entries = bmdp_M.BQ.cardinality * bmdp_M.BQ.cardinality * bmdp_M.AQ.n_u
    sparsity = (1.0 - total_nnz / total_entries) * 100
    print(f"    - Total non-zeros: {total_nnz:,} / {total_entries:,} ({sparsity:.2f}% sparse)")
else:
    print(f"  p_n^{(M)} shape: {bmdp_M.p_n_M.shape}")


In [ ]:
# Verify p_n^{(M)} properties
print("="*70)
print("Verifying p_n^{(M)} properties...")
print("="*70)
print()

cardinality = bmdp_M.BQ.cardinality
n_u = bmdp_M.AQ.n_u

is_sparse = isinstance(bmdp_M.p_n_M, list)

if is_sparse:
    print("✓ Using sparse matrix format (CSR, float16)")
    print()
    
    all_min_vals = []
    all_max_vals = []
    all_row_sums = []
    total_nnz = 0
    
    for k in range(n_u):
        p_n_M_k = bmdp_M.p_n_M[k]
        total_nnz += p_n_M_k.nnz
        
        if p_n_M_k.nnz > 0:
            data = p_n_M_k.data
            all_min_vals.append(float(np.min(data)))
            all_max_vals.append(float(np.max(data)))
            
            row_sums_k = np.array(p_n_M_k.sum(axis=1)).flatten()
            all_row_sums.extend(row_sums_k.tolist())
        else:
            all_row_sums.extend([0.0] * cardinality)
    
    min_val = min(all_min_vals) if all_min_vals else 0.0
    max_val = max(all_max_vals) if all_max_vals else 0.0
    
    print(f"Value range:")
    print(f"  Min: {min_val:.6e}")
    print(f"  Max: {max_val:.6e}")
    if min_val < 0:
        print(f"  ⚠ WARNING: Found negative values!")
    else:
        print(f"  ✓ All values are non-negative")
    
    if max_val > 1.0:
        print(f"  ⚠ WARNING: Found values > 1.0!")
    else:
        print(f"  ✓ All values are ≤ 1.0")
    
    # Check row normalization
    print(f"\nRow normalization (should sum to 1.0 for each (belief, action)):")
    all_row_sums_arr = np.array(all_row_sums)
    min_row_sum = float(np.min(all_row_sums_arr))
    max_row_sum = float(np.max(all_row_sums_arr))
    mean_row_sum = float(np.mean(all_row_sums_arr))
    
    print(f"  Min row sum: {min_row_sum:.6e}")
    print(f"  Max row sum: {max_row_sum:.6e}")
    print(f"  Mean row sum: {mean_row_sum:.6e}")
    
    normalized_count = np.sum(np.abs(all_row_sums_arr - 1.0) < 1e-3)
    print(f"  Normalized rows: {normalized_count}/{len(all_row_sums_arr)}")
    
    if np.allclose(all_row_sums_arr, 1.0, atol=1e-3):
        print(f"  ✓ All rows sum to 1.0 (within tolerance)")
    else:
        print(f"  ⚠ WARNING: Some rows do not sum to 1.0")
    
    # Check sparsity
    total_entries = cardinality * cardinality * n_u
    sparsity = (1.0 - total_nnz / total_entries) * 100
    
    print(f"\nSparsity:")
    print(f"  Non-zero entries: {total_nnz:,} / {total_entries:,}")
    print(f"  Sparsity: {sparsity:.2f}%")

print()
print("="*70)
print("Verification complete!")
print("="*70)
